# Which instances are hard for SSP?

In this notebook, we walk through relating one configuration's cost to public properties of each instance. We use `SSP, scaled` on linear regression, which privately bounds any open coordinate, rescales the data to the unit box and runs sufficient statistics perturbation (SSP). SSP adds noise to `X'X` and `X'y` once, so its noise stays fixed while the statistics grow with the sample size. We relate its cost to:

1. the number of samples per dimension, `n / d`;
2. the condition number of the design.

The configuration is released, so we read the release's ten seeds.

## Opening the linear regression collection

We fetch and open the release as in [Compare on the benchmark](compare_on_the_benchmark.ipynb).

In [ ]:
from pathlib import Path

from dpstats import fetch_benchmark

workspace = fetch_benchmark(Path("dp-benchmark"))

## Choosing the configuration

We take `SSP, scaled` with the standard privacy guarantee. Its steps are the transformations of [Transformations and composition](extend_with_transformations.ipynb).

In [ ]:
from dpstats import open_collection
from dpstats.algorithm import PrivacyInterpretation
from dpstats.tasks import LinearRegression

collection = open_collection(workspace, LinearRegression)
sweep = collection.load_sweep("current")
chosen = [
    algorithm
    for algorithm in collection.load_algorithms(source="release").values()
    if algorithm.privacy_interpretation is PrivacyInterpretation.STANDARD and algorithm.label == "SSP, scaled"
]
weights = collection.load_weights()
print(f"release {collection.release.version}: {len(weights)} instances")
print(repr(chosen[0]))

## Measuring cost

Cost is `prediction_rmse_to_ols` divided by `ols_prediction_se`, the standard error of the OLS prediction, as in [Metrics and baselines](metrics_and_baselines.ipynb); a trial succeeds when its cost is at most 1. A trial that releases nothing costs infinity. We keep ε = 10.

In [ ]:
from dpstats.analysis import measure_results, compute_error_ratio
from dpstats.tasks import list_built_in_metrics

metric = list_built_in_metrics(LinearRegression)["prediction_rmse_to_ols"]
results = collection.load_results(sweep, algorithms=chosen)
trials = compute_error_ratio(
    measure_results(results, metric, collection=collection), "ols_prediction_se", metric=metric
)
at_ten = trials[trials["budget"].map(lambda budget: budget.eps) == 10.0]
results["status"].value_counts()

## Computing the properties

We compute both properties as the task's Explore page does; the condition number reads each instance's Input.

In [ ]:
import numpy as np

properties = collection.instance_features(
    {
        "n / d": lambda instance: instance.metadata.n_samples / instance.metadata.dim,
        "condition number": lambda instance: np.linalg.cond(collection.load_input(instance).X),
    }
)
properties.describe().round(1)

## Plotting cost against samples per dimension

Each point is one instance's median cost over seeds, drawn as the Explore page draws it, with its running median.

In [ ]:
from dpstats.analysis import place_error_ratio_points, plot_error_ratios, trace_running_median

release_weights = weights.set_index("instance_uuid")["weight"]
labels = {
    "ylabel": "prediction RMSE to OLS / ols_prediction_se",
    "point_hover": "{property} {x} · cost {y}",
    "point_beyond_view": "Beyond the current view",
    "count_one": "{count} median point",
    "count": "{count} median points",
    "running_median": "{algorithm}, running median",
}
points = place_error_ratio_points(at_ten, metric=metric, feature=properties["n / d"], weights=release_weights, aggregation="median")
line = trace_running_median(points, feature=properties["n / d"], weights=release_weights, window=0.15, positions=41)
figure = plot_error_ratios(
    points,
    line,
    x_scale="log",
    y_scale="log",
    fence=3.0,
    padding=(0.08, 0.05),
    feature="n / d",
    xlabel="samples per dimension, n / d",
    **labels,
)
figure.show()

The running median falls below 1 once `n / d` is above about 75 and stays there.

## Plotting cost against the condition number

In [ ]:
points = place_error_ratio_points(at_ten, metric=metric, feature=properties["condition number"], weights=release_weights, aggregation="median")
line = trace_running_median(points, feature=properties["condition number"], weights=release_weights, window=0.15, positions=41)
figure = plot_error_ratios(
    points,
    line,
    x_scale="log",
    y_scale="log",
    fence=3.0,
    padding=(0.08, 0.05),
    feature="condition number",
    xlabel="condition number of the design",
    **labels,
)
figure.show()

The running median stays below 1 for condition numbers under about 15 and lies above 1 over most of the range beyond, apart from a dip between about 300 and 500.

## Limits of this comparison

1. The two properties are related: instances with few samples per dimension tend to have larger condition numbers, so neither figure separates them.
2. The condition number is that of the raw design, before the chain rescales it, and is read from the data without privacy. It describes the instances; no private algorithm could read it.
3. The baseline itself shrinks as `n / d` grows, so a falling cost means the private error shrinks faster than the statistical error.
4. A trial whose private bounds fail releases nothing and counts as a failure.